<a href="https://colab.research.google.com/github/saadhana192465019/Digital-Forensics-and-cyber-crime-Investigation--CSA6102/blob/main/Experiment_46_Perceptual_Hashing_and_Watermarking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================
# EXPERIMENT 8
# PERCEPTUAL HASHING AND DIGITAL WATERMARKING
# ==============================================================

import numpy as np
from PIL import Image, ImageDraw, ImageFilter


# ==============================================================
# 1. CREATE ORIGINAL IMAGE
# ==============================================================

def make_original(w=256, h=256, seed=7):

    rng = np.random.default_rng(seed)

    x = np.linspace(0, 1, w)
    y = np.linspace(0, 1, h)

    X, Y = np.meshgrid(x, y)

    r = (200 * X).astype(np.uint8)
    g = (200 * Y).astype(np.uint8)

    b = (
        120 +
        100 * np.sin(12 * X) * np.cos(9 * Y)
    ).astype(np.uint8)

    arr = np.dstack([r, g, b])

    noise = rng.integers(
        -6, 7, arr.shape
    )

    arr = np.clip(
        arr.astype(int) + noise,
        0,
        255
    ).astype(np.uint8)

    img = Image.fromarray(arr, "RGB")

    draw = ImageDraw.Draw(img)

    draw.rectangle(
        [40, 90, 216, 150],
        outline=(255, 255, 255),
        width=4
    )

    draw.text(
        (60, 112),
        "ORIGINAL WORK",
        fill=(255, 255, 255)
    )

    return img


# ==============================================================
# 2. CREATE UNRELATED IMAGE
# ==============================================================

def make_unrelated(w=256, h=256, seed=99):

    rng = np.random.default_rng(seed)

    return Image.fromarray(
        rng.integers(
            0,
            256,
            (h, w, 3),
            dtype=np.uint8
        ),
        "RGB"
    )


# ==============================================================
# 3. AVERAGE HASH (aHash)
# ==============================================================

def ahash(img, size=8):

    gray = np.asarray(
        img.convert("L").resize(
            (size, size),
            Image.LANCZOS
        ),
        dtype=float
    )

    bits = gray > gray.mean()

    return "".join(
        "1" if b else "0"
        for b in bits.flatten()
    )


# ==============================================================
# 4. DIFFERENCE HASH (dHash)
# ==============================================================

def dhash(img, size=8):

    gray = np.asarray(
        img.convert("L").resize(
            (size + 1, size),
            Image.LANCZOS
        ),
        dtype=float
    )

    bits = gray[:, 1:] > gray[:, :-1]

    return "".join(
        "1" if b else "0"
        for b in bits.flatten()
    )


# ==============================================================
# 5. HAMMING DISTANCE
# ==============================================================

def hamming(a, b):

    return sum(
        x != y
        for x, y in zip(a, b)
    )


# ==============================================================
# 6. SAME-WORK DETECTION
# ==============================================================

def is_same_work(img1, img2, threshold=10):

    distance = hamming(
        ahash(img1),
        ahash(img2)
    )

    return distance <= threshold, distance


# ==============================================================
# 7. WATERMARK BIT CONVERSION
# ==============================================================

def _bits(message):

    return "".join(
        f"{byte:08b}"
        for byte in message.encode()
    )


# ==============================================================
# 8. EMBED WATERMARK
# ==============================================================

def embed_watermark(img, message):

    # Add null terminator
    payload = _bits(message) + "0" * 16

    arr = np.array(
        img.convert("RGB")
    )

    # Use blue channel
    flat = arr[:, :, 2].flatten()

    if len(payload) > len(flat):

        raise ValueError(
            "Message too large for this image"
        )

    for i, bit in enumerate(payload):

        flat[i] = (
            flat[i] & 0xFE
        ) | int(bit)

    arr[:, :, 2] = flat.reshape(
        arr[:, :, 2].shape
    )

    return Image.fromarray(
        arr,
        "RGB"
    )


# ==============================================================
# 9. EXTRACT WATERMARK
# ==============================================================

def extract_watermark(
    img,
    max_chars=200
):

    arr = np.array(
        img.convert("RGB")
    )

    flat = arr[:, :, 2].flatten()

    bits = "".join(
        str(int(value) & 1)
        for value in flat[
            :max_chars * 8 + 16
        ]
    )

    output = []

    for i in range(
        0,
        len(bits) - 7,
        8
    ):

        byte = bits[i:i + 8]

        if byte == "00000000":
            break

        output.append(
            chr(int(byte, 2))
        )

    return "".join(output)


# ==============================================================
# 10. PSNR
# ==============================================================

def psnr(a, b):

    x = np.array(
        a,
        dtype=float
    )

    y = np.array(
        b,
        dtype=float
    )

    mse = np.mean(
        (x - y) ** 2
    )

    if mse == 0:

        return float("inf")

    return 10 * np.log10(
        255 ** 2 / mse
    )


# ==============================================================
# 11. RUN EXPERIMENT
# ==============================================================

print("=" * 75)
print("EXPERIMENT 8 - PERCEPTUAL HASHING AND DIGITAL WATERMARKING")
print("=" * 75)


# Create original
original = make_original()

# Create resized copy
resized = (
    original
    .resize((128, 128))
    .resize((256, 256))
)

# Create blurred copy
blurred = original.filter(
    ImageFilter.GaussianBlur(0.7)
)

# Create cropped copy
cropped = (
    original
    .crop((0, 0, 240, 240))
    .resize((256, 256))
)

# Create unrelated image
unrelated = make_unrelated()


# ==============================================================
# 12. PERCEPTUAL HASH RESULTS
# ==============================================================

original_ahash = ahash(original)
resized_ahash = ahash(resized)
unrelated_ahash = ahash(unrelated)

print("\n--- PERCEPTUAL HASHING ---")

print(
    "aHash original :",
    original_ahash
)

print(
    "aHash resized  :",
    resized_ahash
)

print(
    "Hamming distance to resized copy :",
    hamming(
        original_ahash,
        resized_ahash
    )
)

print(
    "Hamming distance to unrelated image :",
    hamming(
        original_ahash,
        unrelated_ahash
    )
)


# ==============================================================
# 13. dHASH RESULTS
# ==============================================================

print("\n--- dHASH ---")

d_original = dhash(original)
d_resized = dhash(resized)
d_unrelated = dhash(unrelated)

print(
    "dHash distance - resized :",
    hamming(
        d_original,
        d_resized
    )
)

print(
    "dHash distance - unrelated :",
    hamming(
        d_original,
        d_unrelated
    )
)


# ==============================================================
# 14. EDITED IMAGE DETECTION
# ==============================================================

print("\n--- EDITED IMAGE DETECTION ---")

same_r, distance_r = is_same_work(
    original,
    resized
)

same_b, distance_b = is_same_work(
    original,
    blurred
)

same_c, distance_c = is_same_work(
    original,
    cropped
)

same_u, distance_u = is_same_work(
    original,
    unrelated
)

print(
    f"Resized copy    : distance={distance_r}, "
    f"same work={same_r}"
)

print(
    f"Blurred copy    : distance={distance_b}, "
    f"same work={same_b}"
)

print(
    f"Cropped copy    : distance={distance_c}, "
    f"same work={same_c}"
)

print(
    f"Unrelated image : distance={distance_u}, "
    f"same work={same_u}"
)


# ==============================================================
# 15. WATERMARKING
# ==============================================================

print("\n--- DIGITAL WATERMARKING ---")

claim = (
    "(c)2026 Cyber Forensics|"
    "CASE/CYB/2026/0417"
)

watermarked = embed_watermark(
    original,
    claim
)

extracted = extract_watermark(
    watermarked
)

print(
    "Original ownership claim :",
    claim
)

print(
    "Extracted ownership claim:",
    extracted
)


# ==============================================================
# 16. PSNR
# ==============================================================

psnr_value = psnr(
    original,
    watermarked
)

print(
    f"PSNR after watermarking : "
    f"{psnr_value:.2f} dB"
)


# ==============================================================
# 17. TEST CASES
# ==============================================================

def run_tests():

    results = []

    # TC1
    results.append((
        "TC1 aHash length is 64 bits",
        len(ahash(original)) == 64
    ))

    # TC2
    results.append((
        "TC2 identical image distance is 0",
        hamming(
            ahash(original),
            ahash(original)
        ) == 0
    ))

    # TC3
    same, d = is_same_work(
        original,
        resized
    )

    results.append((
        f"TC3 resized copy recognised (d={d})",
        same
    ))

    # TC4
    same, d = is_same_work(
        original,
        blurred
    )

    results.append((
        f"TC4 blurred copy recognised (d={d})",
        same
    ))

    # TC5
    same, d = is_same_work(
        original,
        cropped
    )

    results.append((
        f"TC5 cropped copy recognised (d={d})",
        same
    ))

    # TC6
    same, d = is_same_work(
        original,
        unrelated
    )

    results.append((
        f"TC6 unrelated image rejected (d={d})",
        not same
    ))

    # TC7
    results.append((
        "TC7 dHash separates resized and unrelated",
        hamming(
            dhash(original),
            dhash(resized)
        )
        <
        hamming(
            dhash(original),
            dhash(unrelated)
        )
    ))

    # TC8
    results.append((
        "TC8 watermark extracted exactly",
        extract_watermark(
            watermarked
        ) == claim
    ))

    # TC9
    results.append((
        f"TC9 watermark imperceptible "
        f"(PSNR={psnr_value:.1f} dB)",
        psnr_value > 45
    ))

    # TC10
    results.append((
        "TC10 watermark preserves work identity",
        is_same_work(
            original,
            watermarked
        )[0]
    ))

    # TC11
    results.append((
        "TC11 unrelated image has no claim",
        extract_watermark(
            unrelated
        ) != claim
    ))

    # TC12
    try:

        embed_watermark(
            original.resize((8, 8)),
            "X" * 500
        )

        too_big = False

    except ValueError:

        too_big = True

    results.append((
        "TC12 oversized payload rejected",
        too_big
    ))


    # ==========================================================
    # PRINT TEST RESULTS
    # ==========================================================

    print("\n" + "=" * 75)
    print("TEST CASE RESULTS")
    print("=" * 75)

    for name, passed in results:

        print(
            f"{name:<55} -> "
            f"{'PASS' if passed else 'FAIL'}"
        )

    passed_count = sum(
        1
        for _, passed in results
        if passed
    )

    print("\n" + "=" * 75)

    print(
        f"RESULT: {passed_count}/{len(results)} "
        "test cases passed"
    )

    print("=" * 75)

    return passed_count == len(results)


# ==============================================================
# RUN TESTS
# ==============================================================

run_tests()

EXPERIMENT 8 - PERCEPTUAL HASHING AND DIGITAL WATERMARKING

--- PERCEPTUAL HASHING ---
aHash original : 0000000000000000000000110001001101111111011111111111111111111111
aHash resized  : 0000000000000000000000110001001101111111011111111111111111111111
Hamming distance to resized copy : 0
Hamming distance to unrelated image : 30

--- dHASH ---
dHash distance - resized : 0
dHash distance - unrelated : 37

--- EDITED IMAGE DETECTION ---
Resized copy    : distance=0, same work=True
Blurred copy    : distance=0, same work=True
Cropped copy    : distance=2, same work=True
Unrelated image : distance=30, same work=False

--- DIGITAL WATERMARKING ---
Original ownership claim : (c)2026 Cyber Forensics|CASE/CYB/2026/0417
Extracted ownership claim: (c)2026 Cyber Forensics|CASE/CYB/2026/0417
PSNR after watermarking : 78.76 dB

TEST CASE RESULTS
TC1 aHash length is 64 bits                             -> PASS
TC2 identical image distance is 0                       -> PASS
TC3 resized copy recognised (

/tmp/ipykernel_4697/4143488676.py:43: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  img = Image.fromarray(arr, "RGB")
/tmp/ipykernel_4697/4143488676.py:70: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(
/tmp/ipykernel_4697/4143488676.py:195: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(


True